# 03 — Feature Engineering

Build all ML features from the cleaned data, check their distributions and correlation with the target, then save the feature matrix.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from pathlib import Path

def find_project_root(start, depth=5):
    path = start.resolve()
    for _ in range(depth):
        if (path / "src").exists() and (path / "requirements.txt").exists():
            return path
        path = path.parent
    raise RuntimeError(f"Can't find project root from {start}")

PROJECT_ROOT  = find_project_root(Path.cwd())
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR   = PROJECT_ROOT / "outputs" / "figures"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT))
from src.data_loader import load_raw_tables, build_master_df, add_parsed_lap_times
from src.features import build_feature_set, FEATURE_COLUMNS, TARGET_COLUMN

plt.style.use("seaborn-v0_8-darkgrid")
pd.set_option("display.max_columns", 50)
print("Root:", PROJECT_ROOT)

## Load cleaned data

In [ ]:
df = pd.read_csv(PROCESSED_DIR / "master_cleaned.csv")
print(df.shape)
df.head(3)

## Build features

All the logic lives in `src/features.py`. Calling `build_feature_set()` adds every engineered column in the right order.

In [ ]:
df_feat = build_feature_set(df)
print("New columns added:")
new_cols = [c for c in df_feat.columns if c not in df.columns]
print(new_cols)
df_feat[FEATURE_COLUMNS + [TARGET_COLUMN]].head(10)

## NaN check on feature columns

Some NaNs are expected — e.g. a driver's first race has no rolling form yet.

In [ ]:
feat_nulls = df_feat[FEATURE_COLUMNS].isnull().sum()
feat_pct   = (feat_nulls / len(df_feat) * 100).round(1)
pd.DataFrame({"nulls": feat_nulls, "pct%": feat_pct})

## Feature distributions

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(FEATURE_COLUMNS):
    data = df_feat[col].dropna()
    axes[i].hist(data, bins=40, color="steelblue", edgecolor="none", alpha=0.8)
    axes[i].set_title(col, fontsize=9)
    axes[i].set_xlabel("")

plt.suptitle("Feature Distributions", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "feature_distributions.png", dpi=150)
plt.show()

## Correlation with target

Simple point-biserial correlation — how linearly related each feature is to `is_podium`.

In [ ]:
correlations = (
    df_feat[FEATURE_COLUMNS + [TARGET_COLUMN]]
    .corr()[TARGET_COLUMN]
    .drop(TARGET_COLUMN)
    .sort_values()
)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ["#e74c3c" if v < 0 else "#2ecc71" for v in correlations.values]
ax.barh(correlations.index, correlations.values, color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Correlation with is_podium")
ax.set_title("Feature Correlation with Target")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "feature_correlations.png", dpi=150)
plt.show()

print(correlations)

## Podium vs non-podium: grid position

Quick sanity check — does grid position look different for podium finishers?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
df_feat[df_feat[TARGET_COLUMN] == 0]["grid_position"].hist(
    bins=20, alpha=0.6, label="No Podium", color="#e74c3c", ax=ax
)
df_feat[df_feat[TARGET_COLUMN] == 1]["grid_position"].hist(
    bins=20, alpha=0.6, label="Podium", color="#2ecc71", ax=ax
)
ax.set_xlabel("Grid Position")
ax.set_ylabel("Count")
ax.set_title("Grid Position: Podium vs Non-Podium")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "grid_position_by_class.png", dpi=150)
plt.show()

## Save feature matrix

In [ ]:
out_path = PROCESSED_DIR / "features_df.csv"
df_feat.to_csv(out_path, index=False)
print(f"Saved: {out_path}  ({df_feat.shape[0]} rows, {df_feat.shape[1]} cols)")
print(f"Feature columns: {FEATURE_COLUMNS}")
print(f"Target: {TARGET_COLUMN}")